# Copa do Mundo 2026 - Análise Cartola Copa

Análise de desempenho das seleções e jogadores com dados do Cartola Copa.

In [1]:
import glob
import pandas as pd

files = sorted(glob.glob("data/pontuados-rodada-*.csv"))
dfs = []
for f in files:
    df = pd.read_csv(f)
    dfs.append(df)

df_all = pd.concat(dfs, ignore_index=True)
print(f"Total registros: {len(df_all)}")
print(f"Rodadas: {files}")
print(f"Seleções: {df_all['clube_nome'].nunique()}")
print(f"Jogadores: {df_all['atleta_id'].nunique()}")


Total registros: 1358
Rodadas: ['data/pontuados-rodada-1.csv', 'data/pontuados-rodada-2.csv']
Seleções: 48
Jogadores: 863


## Mapa de posições

In [ ]:
POSICOES = {
    1: "Goleiro",
    2: "Lateral",
    3: "Zagueiro",
    4: "Meia",
    5: "Atacante",
    6: "Técnico",
}
df_all["posicao"] = df_all["posicao_id"].map(POSICOES)


## Ranking de Seleções por Pontuação Total

Soma dos pontos de todos os jogadores que entraram em campo, por seleção.

In [ ]:
em_campo = df_all[df_all["entrou_em_campo"] == True]

ranking_selecoes = (
    em_campo.groupby("clube_nome")
    .agg(
        pontuacao_total=("pontuacao", "sum"),
        media_pontuacao=("pontuacao", "mean"),
        jogadores_usados=("atleta_id", "nunique"),
        gols=("G", "sum"),
        assistencias=("A", "sum"),
        jogos=("atleta_id", "count"),
    )
    .sort_values("pontuacao_total", ascending=False)
    .round(2)
)

print("=" * 60)
print("RANKING SELEÇÕES - PONTUAÇÃO TOTAL")
print("=" * 60)
ranking_selecoes.head(20)


## Ranking de Seleções por Média de Pontuação

Média por jogador — mede qualidade técnica independente de quantos jogaram.

In [ ]:
ranking_media = ranking_selecoes.sort_values("media_pontuacao", ascending=False)

print("=" * 60)
print("RANKING SELEÇÕES - MÉDIA POR JOGADOR")
print("=" * 60)
ranking_media.head(20)


## Top 30 Jogadores por Pontuação Total

In [ ]:
top_jogadores = (
    em_campo.groupby(["atleta_id", "apelido", "clube_nome", "posicao"])
    .agg(
        pontuacao_total=("pontuacao", "sum"),
        media_pontuacao=("pontuacao", "mean"),
        rodadas=("pontuacao", "count"),
        gols=("G", "sum"),
        assistencias=("A", "sum"),
    )
    .sort_values("pontuacao_total", ascending=False)
    .round(2)
    .head(30)
)

top_jogadores


## Top Jogadores por Posição

In [ ]:
for pos in ["Goleiro", "Zagueiro", "Lateral", "Meia", "Atacante", "Técnico"]:
    top = (
        em_campo[em_campo["posicao"] == pos]
        .groupby(["apelido", "clube_nome"])
        .agg(
            pontuacao_total=("pontuacao", "sum"),
            gols=("G", "sum"),
            assistencias=("A", "sum"),
            SG=("SG", "sum"),
        )
        .sort_values("pontuacao_total", ascending=False)
        .round(2)
        .head(10)
    )
    print(f"\n{'=' * 50}")
    print(f"TOP 10 - {pos.upper()}")
    print(f"{'=' * 50}")
    display(top)


## Scouts Agregados por Seleção

Perfil técnico: gols, assistências, desarmes, finalizações, faltas.

In [ ]:
scout_cols = ["G", "A", "FT", "FD", "FF", "FS", "FC", "DE", "DS", "SG", "CA", "CV", "GC", "GS"]
available = [c for c in scout_cols if c in em_campo.columns]

scout_labels = {
    "G": "Gols", "A": "Assistências", "FT": "Finalizações Certas",
    "FD": "Finalizações Defendidas", "FF": "Finalizações Fora",
    "FS": "Faltas Sofridas", "FC": "Faltas Cometidas",
    "DE": "Desarmes", "DS": "Dribles", "SG": "Saldo de Gols (DEF)",
    "CA": "Cartões Amarelos", "CV": "Cartões Vermelhos",
    "GC": "Gols Contra", "GS": "Gols Sofridos (GOL)",
}

perfil = (
    em_campo.groupby("clube_nome")[available]
    .sum()
    .fillna(0)
    .astype(int)
    .sort_values("G", ascending=False)
)
perfil.columns = [scout_labels.get(c, c) for c in perfil.columns]

print("PERFIL TÉCNICO POR SELEÇÃO")
perfil.head(20)


## Evolução por Rodada

In [ ]:
rodada_col = [c for c in df_all.columns if 'rodada' in c.lower()]
if not rodada_col:
    df_all["rodada"] = df_all.groupby("atleta_id").cumcount() + 1
    rodada_col = ["rodada"]

# Infer rodada from file order
dfs_labeled = []
for i, f in enumerate(files, 1):
    df_temp = pd.read_csv(f)
    df_temp["rodada"] = i
    dfs_labeled.append(df_temp)

df_rodadas = pd.concat(dfs_labeled, ignore_index=True)
df_rodadas["posicao"] = df_rodadas["posicao_id"].map(POSICOES)
em_campo_r = df_rodadas[df_rodadas["entrou_em_campo"] == True]

evolucao = (
    em_campo_r.groupby(["rodada", "clube_nome"])
    .agg(pontuacao_total=("pontuacao", "sum"))
    .reset_index()
    .pivot(index="clube_nome", columns="rodada", values="pontuacao_total")
    .fillna(0)
)
evolucao["total"] = evolucao.sum(axis=1)
evolucao = evolucao.sort_values("total", ascending=False)

print("PONTUAÇÃO POR RODADA")
evolucao.head(20)


## Rodada 3 — Jogos com Maior Diferença de Ranking FIFA

Confrontos onde a diferença de ranking entre as seleções é maior — oportunidade para escalar jogadores do favorito.

In [ ]:
ranking_fifa = pd.read_csv("data/fifa_ranking.csv")
partidas = pd.read_csv("data/partidas.csv")

rodada3 = partidas[partidas["rodada"] == 3].copy()

rodada3 = rodada3.merge(
    ranking_fifa.rename(columns={"clube_nome": "mandante", "fifa_ranking": "rank_time1", "fifa_points": "pts_time1"}),
    on="mandante",
    how="left",
)
rodada3 = rodada3.merge(
    ranking_fifa.rename(columns={"clube_nome": "visitante", "fifa_ranking": "rank_time2", "fifa_points": "pts_time2"}),
    on="visitante",
    how="left",
)

rodada3["diff_ranking"] = abs(rodada3["rank_time1"] - rodada3["rank_time2"])
rodada3["favorito"] = rodada3.apply(
    lambda r: r["mandante"] if r["rank_time1"] < r["rank_time2"] else r["visitante"], axis=1
)
rodada3["rank_favorito"] = rodada3[["rank_time1", "rank_time2"]].min(axis=1).astype(int)
rodada3["azarão"] = rodada3.apply(
    lambda r: r["visitante"] if r["rank_time1"] < r["rank_time2"] else r["mandante"], axis=1
)
rodada3["rank_azarão"] = rodada3[["rank_time1", "rank_time2"]].max(axis=1).astype(int)

cols = ["favorito", "rank_favorito", "azarão", "rank_azarão", "diff_ranking", "data"]
result = rodada3[cols].sort_values("diff_ranking", ascending=False).reset_index(drop=True)

print("RODADA 3 — JOGOS POR DIFERENÇA DE RANKING FIFA")
print("=" * 70)
result

## Análise da Minha Escalação — Rodada 3

In [ ]:
TITULARES = [
    {"apelido": "Courtois", "posicao": "Goleiro", "selecao": "Bélgica", "preco": 13.66},
    {"apelido": "Nuno Mendes", "posicao": "Lateral", "selecao": "Portugal", "preco": 15.42},
    {"apelido": "Laporte", "posicao": "Zagueiro", "selecao": "Espanha", "preco": 10.39},
    {"apelido": "Van Dijk", "posicao": "Zagueiro", "selecao": "Holanda", "preco": 10.66},
    {"apelido": "Hakimi", "posicao": "Lateral", "selecao": "Marrocos", "preco": 16.34},
    {"apelido": "Saibari", "posicao": "Meia", "selecao": "Marrocos", "preco": 11.24},
    {"apelido": "Olise", "posicao": "Meia", "selecao": "França", "preco": 21.86},
    {"apelido": "Nmecha", "posicao": "Meia", "selecao": "Alemanha", "preco": 14.73},
    {"apelido": "Vini Jr.", "posicao": "Atacante", "selecao": "Brasil", "preco": 21.83},
    {"apelido": "Haaland", "posicao": "Atacante", "selecao": "Noruega", "preco": 27.32},
    {"apelido": "Messi", "posicao": "Atacante", "selecao": "Argentina", "preco": 25.88, "capitao": True},
]

BANCO = [
    {"apelido": "Room", "posicao": "Goleiro", "selecao": "Curaçao", "preco": 7.92},
    {"apelido": "Doué", "posicao": "Meia", "selecao": "França", "preco": 3.91},
    {"apelido": "Riad", "posicao": "Zagueiro", "selecao": "Espanha", "preco": 3.46},
    {"apelido": "H. In-Beom", "posicao": "Meia", "selecao": "Coreia do Sul", "preco": 9.23},
    {"apelido": "Lamine Yamal", "posicao": "Atacante", "selecao": "Espanha", "preco": 21.60},
]

TECNICO = {"apelido": "M. Ouahbi", "selecao": "Marrocos", "preco": 5.33}


def get_adversario(selecao, rodada=3):
    r = partidas[partidas["rodada"] == rodada]
    m = r[(r["mandante"] == selecao) | (r["visitante"] == selecao)]
    if m.empty:
        return "?"
    row = m.iloc[0]
    return row["visitante"] if row["mandante"] == selecao else row["mandante"]


def get_rank(selecao):
    r = ranking_fifa[ranking_fifa["clube_nome"] == selecao]
    return int(r["fifa_ranking"].values[0]) if len(r) > 0 else None


def get_historico(apelido):
    jogador = em_campo[em_campo["apelido"].str.contains(apelido, case=False, na=False)]
    if jogador.empty:
        return {"pts_total": 0, "media": 0, "gols": 0, "assists": 0}
    return {
        "pts_total": round(jogador["pontuacao"].sum(), 2),
        "media": round(jogador["pontuacao"].mean(), 2),
        "gols": int(jogador["G"].sum()),
        "assists": int(jogador["A"].sum()),
    }


rows = []
for p in TITULARES + BANCO:
    h = get_historico(p["apelido"])
    adv = get_adversario(p["selecao"])
    tr = get_rank(p["selecao"])
    ar = get_rank(adv) if adv != "?" else None
    diff = (ar - tr) if tr and ar else None
    rows.append({
        "jogador": p["apelido"],
        "posicao": p["posicao"],
        "selecao": p["selecao"],
        "fifa_rank": tr,
        "adversario": adv,
        "adv_rank": ar,
        "diff_rank": diff,
        "media_cartola": h["media"],
        "pts_total": h["pts_total"],
        "gols": h["gols"],
        "assists": h["assists"],
        "preco": p["preco"],
        "titular": p in TITULARES,
        "capitao": p.get("capitao", False),
    })

df_esc = pd.DataFrame(rows)

print("TITULARES")
print("=" * 70)
display(df_esc[df_esc["titular"]][
    ["jogador", "posicao", "selecao", "fifa_rank", "adversario", "adv_rank", "diff_rank", "media_cartola", "pts_total", "gols", "assists", "capitao"]
])

print("\nBANCO")
print("=" * 70)
display(df_esc[~df_esc["titular"]][
    ["jogador", "posicao", "selecao", "fifa_rank", "adversario", "adv_rank", "diff_rank", "media_cartola", "pts_total"]
])

In [ ]:
selecoes_usadas = df_esc["selecao"].unique()
custo_tit = sum(p["preco"] for p in TITULARES)
custo_banco = sum(p["preco"] for p in BANCO)
custo_total = custo_tit + custo_banco + TECNICO["preco"]

print("RESUMO DA ESCALAÇÃO")
print("=" * 70)
print(f"  Formação: 4-3-3")
print(f"  Seleções: {len(selecoes_usadas)} diferentes")
print(f"  Capitão: Messi (pontua em dobro)")
print(f"  Técnico: {TECNICO['apelido']} ({TECNICO['selecao']})")
print(f"  Custo: C$ {custo_total:.2f} (titulares: {custo_tit:.2f} | banco: {custo_banco:.2f} | técnico: {TECNICO['preco']:.2f})")

print(f"\nANÁLISE DE RISCO POR CONFRONTO")
print("=" * 70)
confrontos = {}
for p in TITULARES:
    adv = get_adversario(p["selecao"])
    key = f"{p['selecao']} vs {adv}"
    if key not in confrontos:
        tr = get_rank(p["selecao"])
        ar = get_rank(adv) if adv != "?" else None
        confrontos[key] = {"jogadores": [], "team_rank": tr, "adv_rank": ar}
    confrontos[key]["jogadores"].append(p["apelido"])

for conf, info in sorted(confrontos.items(), key=lambda x: len(x[1]["jogadores"]), reverse=True):
    n = len(info["jogadores"])
    tr, ar = info["team_rank"], info["adv_rank"]
    try:
        diff = int(ar) - int(tr)
        risk = "BAIXO ✅" if diff > 20 else "MÉDIO ⚠️" if diff > 0 else "ALTO 🔴"
    except (ValueError, TypeError):
        risk = "?"
    print(f"\n  {conf} (#{tr} vs #{ar}) — Risco: {risk}")
    print(f"    Jogadores: {', '.join(info['jogadores'])} ({n})")
    if n >= 3:
        print(f"    ⚠️  {n} jogadores no mesmo jogo — alta concentração")